In [1]:
import os
import pickle
import gc
from tqdm import tqdm
import time
import polars as pl
import numpy as np
import shutil
import glob
from scipy import sparse
from tienlen_libs import Card, Utils, ActionGenerator, NaturalWin
from tienlen_libs.constant import ComboType
import math
from collections import defaultdict

In [ ]:
def divide_all_files_data_into_3_part(all_files_data):
    print("Before shuffling the all_files_data")
    for i in range(5): print(all_files_data[i])
    np.random.shuffle(all_files_data)
    print("After shuffling the all_files_data")
    for i in range(5): print(all_files_data[i])
    #Divide the all_files_data into 3 parts: Part 1: 70%, Part 2: 20%, Part 3: 10%
    pos_part_1 = int(len(all_files_data)*0.7)
    pos_part_2 = int(len(all_files_data)*0.9)
    assert pos_part_1 < pos_part_2 < len(all_files_data)
    part_train = all_files_data[:pos_part_1]
    part_validation = all_files_data[pos_part_1:pos_part_2]
    part_test = all_files_data[pos_part_2:]
    print("The length of all_files_data: {:,}".format(len(all_files_data)))
    print("The length of part_train: {:,}".format(len(part_train)))
    print("The length of part_validation: {:,}".format(len(part_validation)))
    print("The length of part_test: {:,}".format(len(part_test)))
    assert len(part_train) + len(part_validation) + len(part_test) == len(all_files_data)
    return part_train, part_validation, part_test

def main_gen_train_validation_test_folder(data_all_dir, 
                                          data_train_dir, 
                                          data_validation_dir, 
                                          data_test_dir):
    for directory in [data_train_dir, data_validation_dir, data_test_dir]:
        os.makedirs(directory, exist_ok=True)

    all_files_data = glob.glob(os.path.join(data_all_dir, '*'))
    assert len(all_files_data) == len(os.listdir(data_all_dir))
    part_train, part_validation, part_test = divide_all_files_data_into_3_part(all_files_data)
    
    for part, part_dir in zip([part_train, part_validation, part_test], 
                            [data_train_dir, data_validation_dir, data_test_dir]):
        print(f"\nCopy the files to the {part_dir} folder...........")
        for src_dir in tqdm(part):
            file_name = os.path.basename(src_dir)
            des_dir = os.path.join(part_dir, file_name)
            assert not os.path.exists(des_dir)
            shutil.copy2(src_dir, des_dir)
            # shutil.copytree(src_dir, des_dir)

def divide_src_folder_into_multiple_small_folder(src_folder, files_per_small_folder=1000):
    print(f"\nSplitting the {src_folder} into multiple small folder")
    all_files_data = list(glob.glob(os.path.join(src_folder, '*.pkl')))
    np.random.shuffle(all_files_data)
    total_files = len(all_files_data)
    n_small_folder = math.ceil(total_files / files_per_small_folder)
    
    for i in range(n_small_folder):
        small_folder_name = src_folder + f'_{i+1}'
        os.makedirs(small_folder_name)
        
        start_idx = i * files_per_small_folder
        end_idx = min(start_idx + files_per_small_folder, total_files)
        print(f"Moving files from index {start_idx} : {end_idx-1} to {small_folder_name}")
        for file_dir in all_files_data[start_idx:end_idx]:
            shutil.move(file_dir, small_folder_name)

src_folder = r'D:\Project\TLMN\preprocess'
all_preprocess_folders = glob.glob(os.path.join(src_folder, '*'))
for elem in all_preprocess_folders: print(elem)
for preprocess_folder in all_preprocess_folders:
    print(f"\nDivide the preprocess_folder {preprocess_folder} into train, validation, test folder")
    data_all_dir = preprocess_folder
    data_train_dir = r"D:\Project\TLMN\preprocess\data_train"
    data_validation_dir = r"D:\Project\TLMN\preprocess\data_validation"
    data_test_dir = r"D:\Project\TLMN\preprocess\data_test"

    main_gen_train_validation_test_folder(data_all_dir, data_train_dir, data_validation_dir, data_test_dir)          

In [ ]:
def concat_to_a_compact_file(dataset_type, start_idx, end_idx):
    for i in range(start_idx, end_idx + 1, 2):
        print(f"\nProcessing the {i} and {i+1} files")
        data_1 = pl.read_parquet(fr"F:\Q_table_{dataset_type}_{i}.parquet")
        print(f"The number of rows: {len(data_1):,}")
        data_2 = pl.read_parquet(fr"F:\Q_table_{dataset_type}_{i+1}.parquet")
        print(f"The number of rows: {len(data_2):,}")
        data_concat = pl.concat([data_1, data_2])
        print(f"The number of rows: {len(data_concat):,}")
        data_concat.write_parquet(fr"F:\compact_data\Q_table_{dataset_type}_{i}_{i+1}.parquet")

concat_to_a_compact_file("train", 1, 14)
concat_to_a_compact_file("validation", 1, 4)
concat_to_a_compact_file("test", 1, 2)

In [ ]:
def check_range_value_of_a_series(column_series, min_value=0, max_value=1):
    assert (min_value <= column_series).all() and (column_series <= max_value).all(), \
        f"The range value of the column {column_series.name} is not correct"

def check_range_value_after_min_max_scale(dataset):
    for col_name in dataset.columns:
        # if col_name in ["average_point"]:
        #     continue
        check_range_value_of_a_series(dataset[col_name])

def normalize_dataset(dataset):
    bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
    dataset = dataset.with_columns(
        [
            pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Int8
            (pl.col("player1_action_max_card") + 1) / 52,  # Normalize player1_action_max_card
            (pl.col("action_max_card") + 1) / 52,  # Normalize action_max_card
        ]
    )
    return dataset

src_data = r"F:\compact_data"
batch_size = 2048
folder_parent_dir = r"F:\big_data"

if not os.path.exists(folder_parent_dir):
    os.makedirs(folder_parent_dir)

for dataset_type in ["train"]:
    count = 0
    folder_dataset_type = os.path.join(folder_parent_dir, f"data_{dataset_type}")
    os.makedirs(folder_dataset_type)
    for index in range(1, 15, 2):
        print(f"\nLoading the {dataset_type} data of Q_table_{dataset_type}_{index}_{index+1}.parquet......")
        dataset_dir = fr"{src_data}\Q_table_{dataset_type}_{index}_{index+1}.parquet"
        dataset = pl.read_parquet(dataset_dir)
        print(f"The shape of the dataset: ({dataset.shape[0]:,}; {dataset.shape[1]})")
        print(dataset.head())
        dataset = dataset.sample(fraction=1.0, shuffle=True)
        print(dataset.head())
        print("Shuffled data: DONE")
        for i in range(0, len(dataset), batch_size):
            batch_data = dataset[i : i+batch_size]
            batch_data = normalize_dataset(batch_data)
            check_range_value_after_min_max_scale(batch_data)
            batch_data_to_numpy = batch_data.to_numpy()
            batch_data_csr = sparse.csr_matrix(batch_data_to_numpy)
            sparse.save_npz(os.path.join(folder_dataset_type, f"data_{dataset_type}_batch_{count}.npz"), batch_data_csr)
            count += 1
        del dataset; gc.collect()

In [ ]:
import os
import pickle
import glob
from tqdm import tqdm

src_data = r"D:\Project\TLMN\data_13_11"
pkl_files = glob.glob(os.path.join(src_data, '**/state_*.pkl'), recursive=True)
set_len_player1_card = set()
for file in tqdm(pkl_files):
    with open(file, 'rb') as f:
        Q_table = pickle.load(f)
    assert len(Q_table) == 1
    for state in Q_table:
        set_len_player1_card.add(state[-1])
print(set_len_player1_card)

In [ ]:
import dask.dataframe as dd
import os

data_train_dir = r"D:\Project\TLMN\preprocess\data_train"
data_validation_dir = r"D:\Project\TLMN\preprocess\data_validation"
data_test_dir = r"D:\Project\TLMN\preprocess\data_test"
for data_dir in [data_train_dir, data_validation_dir]:
    print(f"\nConcatenating the data_dir {data_dir}")
    df = dd.read_parquet(os.path.join(data_dir, '*.parquet'))
    print(df)
    df_computed = df.compute()
    print("The shape of df_computed is:", df_computed.shape)
    df_computed.to_parquet(data_dir + ".parquet", index=False)